# 5-minute Jammi AI quickstart — register, embed, search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/recipes/quickstart.ipynb)

Built from [`cookbook/quickstart/quickstart.py`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/quickstart/quickstart.py). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

Walks the four steps from `cookbook/quickstart/`'s README:

1. `jammi.connect("file://…")` — open a local in-process session
2. `db.add_source` — attach the tiny corpus fixture
3. `db.generate_embeddings(..., modality="text")` — build a 32-dim USEARCH-backed index
4. `db.encode_query(...)` + `db.search(...)` — execute a similarity query (returns a table)

`connect(target)` is the one front door: a `file://` target runs the engine
in-process; flipping to a `https://` / `grpc://` target — no code change —
talks to a remote server via the `jammi-ai` client.

Uses the local `cookbook/fixtures/tiny_bert` encoder so the script runs
without network access. Swap `MODEL` for a Hugging Face Hub model ID like
`sentence-transformers/all-MiniLM-L6-v2` for production.

Exits 0 in well under 30 seconds on CPU.

In [ ]:
from __future__ import annotations

import os
import tempfile
from pathlib import Path

# Pin the engine to CPU so the example is reproducible on any machine. Engine
# tuning (device, batch size, memory) is configuration, read from the
# environment — `connect(target)` itself takes only the target.
os.environ.setdefault("JAMMI_GPU__DEVICE", "-1")
os.environ.setdefault("JAMMI_ENGINE__BATCH_SIZE", "8")

import jammi
from jammi_cookbook import fixtures

CORPUS_PATH = fixtures.path("tiny_corpus.parquet")
MODEL = fixtures.model("tiny_bert")


def main() -> int:
    # 1. Connect to a local, in-process engine rooted at the temp dir. The session
    #    is a context manager, and block exit CLOSES it — the embedded engine
    #    holds its catalog until close() returns, so it must be released before
    #    the directory is removed (the `with` items unwind in reverse order).
    with tempfile.TemporaryDirectory() as tmp, jammi.connect(f"file://{tmp}") as db:
        # 2. Register the tiny corpus as a Parquet source.
        db.add_source("corpus", url=str(CORPUS_PATH), format="parquet")

        # 3. Build a 32-dim embedding table over the `content` column.
        db.generate_embeddings(
            source="corpus",
            model=MODEL,
            columns=["content"],
            key="id",
            modality="text",
        )

        # 4. Encode a query and run a top-3 similarity search.
        query_vec = db.encode_query(model=MODEL, query="how does quantum computing work?")
        results = db.search("corpus", query=query_vec, k=3)  # pyarrow.Table

        rows = results.to_pylist()
        if not rows:
            raise RuntimeError("quickstart returned zero rows")

        print("id        similarity  title")
        for row in rows:
            print(
                f"{row['_row_id']:<8}  {row['similarity']:>9.4f}  {row['title']}"
            )

    return 0

In [ ]:
assert main() == 0